In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
# # Text data
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("data/Python.txt", encoding="utf-8")

# document = loader.load()
# # document

In [4]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# # document

# Ingestion Pipeline

In [5]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

C:\Users\Syed khizer\AppData\Local\Temp\ipykernel_1368\451959863.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


# Data => Langchain Documents

In [6]:
def load_all_pdfs():
    folder_path="data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("Total pdf: ", num_docs)
    print("Total pages: ", len(all_docs))
    return all_docs

In [7]:
all_pdf_documents = load_all_pdfs()

Total pdf:  2
Total pages:  32


# Chunks

In [8]:
# chunks
# !pip install langchain_text_splitters

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [10]:
chunks = split_docs(all_pdf_documents)
len(chunks)

320

# Embeddings 

In [11]:
from sentence_transformers import SentenceTransformer

In [12]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name=model_name
        print("Loading model....",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("Embedding dimensions= ",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embedding shape: ",embeddings.shape)
        return embeddings

In [13]:
embedding_manager = EmbeddingManager()

Loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions=  384


C:\Users\Syed khizer\AppData\Local\Temp\ipykernel_1368\3203930644.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions= ",self.model.get_sentence_embedding_dimension())


# Vector Store

In [14]:
import chromadb
import uuid

In [15]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        # Create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        #create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("Initialized the vector store with collection: ", self.collection_name)
        print("Docs in collection: ", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of Docs does not match number of embedings")

        # store => ids, embedding, document, metadata 
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("Total documents added in vector store: ",len(documents_content))
        print("Documents in collection: ",self.collection.count())

In [16]:
vector_store = VectorStoreManager()

Initialized the vector store with collection:  pdf_documents
Docs in collection:  1280


In [17]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embedding shape:  (320, 384)
Total documents added in vector store:  320
Documents in collection:  1600


# Retrieval Pipeline

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [19]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents  = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })

            print(f"Retrieved {len(retrieved_docs)} documents")

        else:
            print("No doc found!")

        return retrieved_docs

In [20]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [21]:
rag_retriever.retrieve("What is decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
Retrieved 5 documents


[{'id': 'doc_404b4fe4-f485-4b16-9ad3-851e0b74a389',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'publisher': 'Curran Associates, Inc.',
   'title': 'Attention is All you Need',
   'date': '2017',
   'language': 'en-US',
   'book': 'Advances in Neural Information Processing Systems 30',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett',
   'type': 'Conference Proceedings',
   'content_length': 447,
   'created': '2017',
   'description': 'Paper accepted and presented at the Neural Infor

# Integrate with LLMs

# Google GEMINI

In [22]:
# %pip install -U langchain-google-genai python-dotenv

In [23]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    google_api_key=os.getenv("GEMINI_API"),
    model="gemini-2.5-flash-lite",
    temperature=0.1,
    max_tokens=1024
)

In [24]:
# generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, tok_k=3):
    results = rag_retriever.retrieve(query, tok_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # prompt = context + query    
    prompt = f""" use given context to generate the answer
                Context: {context}
                Query: {query}"""

    response = llm.invoke(prompt)
    return response.content

In [25]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


embedding shape:  (1, 384)
Retrieved 3 documents


In [26]:
print(answer)

Based on the provided context, RAG refers to **Retrieval-Augmented Generation** methods. The text mentions a "thorough and systematic review of the state-of-the-art RAG methods" and delineates its evolution through paradigms including "naive RAG."


# Groq

In [27]:
# %pip install -U langchain-groq python-dotenv

In [28]:
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    api_key=os.getenv("GROQ_API"),
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [29]:
# generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, tok_k=3):
    results = rag_retriever.retrieve(query, tok_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # prompt = context + query    
    prompt = f""" use given context to generate the answer
                Context: {context}
                Query: {query}"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [30]:
answer = generate_output("what is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape:  (1, 384)
Retrieved 3 documents


In [31]:
print(answer)

**RAG (Retrieval‑Augmented Generation)** is a hybrid approach that combines two key components:

1. **Retrieval** – A search or indexing module (often based on dense or sparse vector similarity) fetches relevant pieces of external information (documents, passages, tables, etc.) from a large knowledge base or corpus in response to a user query.

2. **Generation** – A generative language model (e.g., GPT‑4, LLaMA, T5) takes the retrieved text as additional context and produces a fluent, grounded answer or continuation.

The idea is to give the language model access to up‑to‑date or domain‑specific facts that it may not have memorized during pre‑training, thereby improving factual accuracy, reducing hallucinations, and enabling the system to handle queries that require recent or specialized knowledge.

Typical RAG pipelines work as follows:

1. **Query encoding** – The user’s input is encoded into a vector.
2. **Document retrieval** – The vector is used to retrieve the top‑k most relevant